# Eye diagram

MS-PRS $L_0=3$ balanced on one quadrature. The waveform comes from the simulator: modulate, RRC shape, noise at the commanded Eb/N0, matched filter.

In [ ]:
import subprocess, pathlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

ROOT = pathlib.Path.cwd().parent
MSPRS = ROOT / "build" / "bin" / "msprs"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

L0, FAMILY, EB_N0_DB = 3, "balanced", 15.11

PHOSPHOR = {"screen": "#06110a", "grid": "#1c4426", "axis": "#3d8c4d",
            "trace": "#3dff63", "ideal": "#e6ffe0", "text": "#7dff9a",
            "bezel": "#1a1f1b"}

In [ ]:
out = subprocess.run(
    [str(MSPRS), "--mode", "eye", "--L0", str(L0), "--family", FAMILY,
     "--ebn0", str(EB_N0_DB), "--sps", "32", "--symbols", "3000"],
    capture_output=True, text=True, check=True).stdout.splitlines()

hdr = out[0].split()
sps, delay = int(hdr[2]), int(hdr[4])
rx = np.array([float(v) for v in out[1:]])

span = 2 * sps                      # two symbol periods across the screen
starts = delay + np.arange(12, 3000 - 12) * sps - sps
windows = np.stack([rx[s:s + span + 1] for s in starts])
t = np.linspace(-1.0, 1.0, span + 1)
print(f"sps {sps}, {len(windows)} traces")

In [ ]:
# A digital-phosphor scope accumulates how often the beam crosses each cell.
# With ten amplitude levels an alpha-blended overlay either saturates or
# disappears; density shows the common paths and the rare ones at once.
nt, na = 900, 620
ylim = 1.08 * float(np.max(np.abs(windows)))
fine = np.linspace(-1.0, 1.0, nt)
interp = np.stack([np.interp(fine, t, w) for w in windows])

hits, _, _ = np.histogram2d(np.tile(fine, len(windows)), interp.ravel(),
                            bins=[nt, na], range=[[-1.0, 1.0], [-ylim, ylim]])
glow = (hits.T / hits.max()) ** 0.62

screen = LinearSegmentedColormap.from_list("phosphor", [
    (0.00, PHOSPHOR["screen"]), (0.30, "#0a2b14"), (0.62, "#1f8c3a"),
    (0.88, PHOSPHOR["trace"]), (1.00, PHOSPHOR["ideal"])])

fig, ax = plt.subplots(figsize=(5.6, 3.7))
fig.patch.set_facecolor(PHOSPHOR["bezel"])
ax.set_facecolor(PHOSPHOR["screen"])
ax.imshow(glow, origin="lower", aspect="auto", cmap=screen,
          extent=[-1.0, 1.0, -ylim, ylim], interpolation="bilinear", zorder=1)
ax.set(xlim=(-1.0, 1.0), ylim=(-ylim, ylim), xticks=[], yticks=[])
ax.grid(False)

for k in range(1, 8):
    ax.axvline(-1.0 + k * 0.25, color=PHOSPHOR["grid"], lw=0.5, alpha=0.55, zorder=4)
    ax.axhline(-ylim + k * 2 * ylim / 8, color=PHOSPHOR["grid"], lw=0.5, alpha=0.55, zorder=4)
ax.axhline(0.0, color=PHOSPHOR["axis"], lw=0.9, alpha=0.7, zorder=4)
ax.axvline(0.0, color=PHOSPHOR["axis"], lw=0.9, alpha=0.7, zorder=4)
minor_x = [-1.0 + k * 0.05 for k in range(1, 40) if k % 5]
minor_y = [-ylim + k * 2 * ylim / 40 for k in range(1, 40) if k % 5]
ax.vlines(minor_x, -0.022 * ylim, 0.022 * ylim, color=PHOSPHOR["axis"], lw=0.6, zorder=4)
ax.hlines(minor_y, -0.009, 0.009, color=PHOSPHOR["axis"], lw=0.6, zorder=4)

text = dict(color=PHOSPHOR["text"], family="monospace", zorder=6,
            bbox=dict(facecolor=PHOSPHOR["screen"], edgecolor="none", alpha=0.72, pad=1.8))
ax.text(-0.97, ylim * 0.93, f"{FAMILY.upper()}  L0={L0}  Eb/N0 {EB_N0_DB:.1f} dB",
        ha="left", va="top", fontsize=7, **text)
ax.text(0.97, 0.03 * ylim, "t/T", ha="right", va="bottom", fontsize=9, **text)
ax.text(-0.97, -ylim * 0.93, "A", ha="left", va="bottom", fontsize=9, **text)
for spine in ax.spines.values():
    spine.set(color=PHOSPHOR["bezel"], linewidth=3.0)

for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"eye_msprs_L{L0}_{FAMILY}.{ext}", dpi=200,
                facecolor=fig.get_facecolor(), bbox_inches="tight")